# GeFolki quickstart

Dense optical-flow coregistration with the `gefolki` package, using the sample data in
`../datasets` (download it first: `python datasets/fetch.py` from the repository root):

1. EFolki on two optical images (QuickBird / WorldView)
2. GeFolki on radar / optical (contrast inversions between sensors)
3. Accuracy: end-point error (EPE) against the EvalGeFolki ground-truth flows
4. `register()` on a georeferenced pair (files in, registered file out)
5. The same from the command line

Extra packages for this notebook: `pip install matplotlib tifffile` (plus `jupyter`).
Run it from the `examples/` directory of a repository checkout.

Flow convention everywhere: `slave(x + u, y + v) ≈ master(x, y)`; `u` is the column (x)
shift and `v` the row (y) shift in pixels, and `gefolki.warp(slave, u, v)` gives the
registered slave on the master grid.

In [ ]:
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rasterio.errors
import tifffile

import gefolki as g

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)  # PNGs
DATA = Path("..") / "datasets"
print(g.__version__, "- default device:", g.backend_info()["default"])


def read(path, band=1):
    with rasterio.open(path) as ds:
        return ds.read(band).astype(np.float32)


def stretch(a, mask=None):
    lo, hi = np.percentile(a[mask] if mask is not None else a, (1, 99))
    return np.clip((a - lo) / (hi - lo), 0, 1)


def fuse(master, slave):
    """Magenta/green overlay: aligned structures look grey."""
    m, s = stretch(master), stretch(slave)
    return np.dstack([s, m, s])


def show(images, titles, **kw):
    fig, axes = plt.subplots(1, len(images), figsize=(4.5 * len(images), 4.5))
    for ax, im, t in zip(np.atleast_1d(axes), images, titles, strict=True):
        h = ax.imshow(im, **kw)
        ax.set_title(t)
        ax.axis("off")
        if im.ndim == 2:
            fig.colorbar(h, ax=ax, shrink=0.7)
    plt.tight_layout()
    plt.show()

## 1. EFolki: optical / optical

QB was affinely pre-registered to WV with manual control points; residual shifts remain.
EFolki matches rank-filtered images, so it tolerates monotonic radiometric differences.
`mask` marks valid pixels (here: WV > 0).

In [ ]:
wv, qb = read(DATA / "WV.tif"), read(DATA / "QB.tif")
mask = wv > 0
u, v = g.efolki(wv, qb, mask=mask, levels=5, radius=(16, 8), iterations=4)
qb_reg = g.warp(qb, u, v)
print(f"median |flow| {np.median(np.hypot(u, v)[mask]):.2f} px")

crop = np.s_[600:1100, 1000:1500]
show([np.hypot(u, v)], ["|flow| (px)"], cmap="viridis")
show([fuse(wv[crop], qb[crop]), fuse(wv[crop], qb_reg[crop])], ["before", "after"])

## 2. GeFolki: radar / optical

P-band SAR (Pauli red channel) and an optical orthophoto. `gefolki` adds a local
contrast-inversion test (CLAHE) to EFolki. The defaults (6 levels, radii 32..8, 2
iterations, rank 4) are those of the GeFolki manual.

In [ ]:
radar = read(DATA / "radar_bandep.png", 1)
optical = read(DATA / "optiquehr_georef.png", 2)
u, v = g.gefolki(radar, optical)
optical_reg = g.warp(optical, u, v)
print(f"median |flow| {np.median(np.hypot(u, v)):.2f} px")

crop = np.s_[700:1300, 700:1300]
show([np.hypot(u, v)], ["|flow| (px)"], cmap="viridis")
show([fuse(radar[crop], optical[crop]), fuse(radar[crop], optical_reg[crop])], ["before", "after"])

## 3. End-point error on EvalGeFolki

Warp each master by its ground-truth flow (`slave(x) = master(x + gt)`), then recover the
flow; the estimate should be `-gt`. EPE = mean `|flow + gt|` (30 px border excluded).
For reference, the legacy EFolki gets 0.10 px (S1S2) and 0.37 px (HR) here.
The ground-truth files hold two pages, so read them with `tifffile`.

In [ ]:
cases = {
    "S1S2": ("S1S2/S1_patch11.tif", "S1S2/Flow_patch11.tif"),
    "HR": (
        "HR/lngley_05521_09059_000_090813_L090_CX_01_pauli_x016_y020_radar.tif",
        "HR/lngley_05521_09059_000_090813_L090_CX_01_pauli_x016_y020_flow.tif",
    ),
}
inner = np.s_[30:-30, 30:-30]
for name, (img, flow) in cases.items():
    master = read(DATA / "EvalGeFolki" / img)
    gt = tifffile.imread(DATA / "EvalGeFolki" / flow)  # (H, W, 2): u, v
    slave = g.warp(master, gt[..., 0], gt[..., 1])
    for fn in (g.efolki, g.gefolki):
        u, v = fn(master, slave, levels=3, radius=(16, 8), iterations=4)
        epe = np.hypot(u + gt[..., 0], v + gt[..., 1])[inner].mean()
        print(f"{name:5s} {fn.__name__:8s} mean EPE {epe:.3f} px")

## 4. `register()` on a georeferenced pair

`register(master, slave, output)` reads the master resampled onto the slave grid, builds
one registration band per image, computes the flow, warps **all** slave bands and writes
them with the slave's grid, CRS, dtype, nodata and band metadata. Below, the slave is the
S1 patch (2 bands, EPSG:32631) distorted by the ground-truth flow and written as a new
GeoTIFF; registering it back to the original should give flow ≈ `-gt`.

In [ ]:
tmp = Path(tempfile.mkdtemp())
s1 = DATA / "EvalGeFolki" / "S1S2" / "S1_patch11.tif"
gt = tifffile.imread(DATA / "EvalGeFolki" / "S1S2" / "Flow_patch11.tif")

with rasterio.open(s1) as src:
    profile, bands = src.profile, src.read()
with rasterio.open(tmp / "slave.tif", "w", **profile) as dst:
    dst.write(g.warp(bands, gt[..., 0], gt[..., 1]))

params = g.FlowParams(levels=3, radius=(16, 8), iterations=4, rank=4)
res = g.register(
    s1, tmp / "slave.tif", tmp / "registered.tif", params=params, flow_output=tmp / "flow.tif"
)
print(res.output, res.flow_stats, {k: round(t, 3) for k, t in res.timings.items()})

with rasterio.open(res.flow_path) as f:
    u, v = f.read()
    epe = np.hypot(u + gt[..., 0], v + gt[..., 1])[inner].mean()
    print(f"flow CRS: {f.crs} | EPE vs -gt: {epe:.3f} px")
with rasterio.open(res.output) as r:
    print("registered:", r.count, "bands", r.dtypes[0], r.crs, r.transform == profile["transform"])

Parameter presets (`g.PRESETS`) cover common sensor pairs; pass `preset="optical-sar"` etc.
instead of `params`. `method="efolki"` / `"gefolki"` / `"folki"` switches only the
variant. For a hyperspectral line and an RGB ortho the defaults pick the
`hyperspectral-rgb` preset, RGB luminance for the master and the 500-600 nm band mean for
the slave; see `register_hyperspectral.py` for a per-flight-line batch script.

In [ ]:
for name, p in g.PRESETS.items():
    print(f"{name:18s} {p}")

## 5. Command line

The same steps with the `gefolki` CLI (`gefolki --help`, `gefolki COMMAND --help`):

```bash
gefolki register MASTER SLAVE OUT.tif --preset optical-sar --flow-output FLOW.tif
gefolki flow MASTER SLAVE FLOW.tif --levels 3 --radius 16,8 --iterations 4
gefolki warp SLAVE FLOW.tif OUT.tif
gefolki locate BIG.tif CHIP.png --chip-output match.tif
gefolki inspect RASTER --bands
gefolki presets
gefolki info
```

Below they run through `python -m gefolki` so the notebook's interpreter is used.

In [ ]:
gf = f"{sys.executable} -m gefolki"
slave, flow, out = tmp / "slave.tif", tmp / "flow_cli.tif", tmp / "registered_cli.tif"
!{gf} flow {s1} {slave} {flow} --levels 3 --radius 16,8 --iterations 4 --json
!{gf} warp {slave} {flow} {out} --quiet
!{gf} inspect {out}
!{gf} locate {DATA/"S1_Jacksonville_GEE.tif"} {DATA/"JacksonvilleNavalAirStation_sandiaKu.png"}